# Elastic Net Regression — Mini-Batch Gradient Descent with Momentum

## 1. Problem Setup

Assume we have a dataset

$$
\{(x_i,y_i)\}_{i=1}^{N}
$$

where

$$
x_i \in \mathbb{R}^D
$$

is the feature vector and

$$
y_i \in \mathbb{R}
$$

is the continuous target value.

The goal is to learn a model that predicts $y$ from $x$ while controlling model complexity using both L1 and L2 regularization.

---

## 2. Linear Model

The model learns a weight vector

$$
w \in \mathbb{R}^D
$$

The prediction is

$$
\hat{y} = w^T x
$$

---

## 3. Adding the Bias Term

To include an intercept, we augment the feature vector

$$
x =
\begin{bmatrix}
1 \\
x
\end{bmatrix}
$$

The weight vector becomes

$$
w =
\begin{bmatrix}
b \\
w
\end{bmatrix}
$$

Thus the prediction becomes

$$
\hat{y} = w^T x
$$

---

## 4. Objective Function

Elastic Net combines L1 and L2 regularization:

$$
L(w) =
\frac{1}{2N} \|Xw - y\|_2^2
+ \lambda_1 \|w\|_1
+ \frac{\lambda_2}{2} \|w\|_2^2
$$

where:

- $\lambda_1$ controls sparsity (Lasso)
- $\lambda_2$ controls weight shrinkage (Ridge)

---

## 5. Gradient of Loss

The gradient of the differentiable part is

$$
\nabla_w L =
\frac{1}{N} X^T (Xw - y) + \lambda_2 w
$$

Note: L1 term is handled separately via **soft-thresholding**.

---

## 6. Mini-Batch Gradient Descent

- Shuffle the dataset each epoch  
- Split into mini-batches of size $B$  
- Compute gradient on each batch  

$$
\nabla_w^{(batch)} =
\frac{1}{B} X_{batch}^T (X_{batch} w - y_{batch}) + \lambda_2 w
$$

---

## 7. Momentum Update

Momentum is used to stabilize updates:

$$
m = \beta m + (1 - \beta)\nabla_w
$$

Weights are updated as:

$$
w = w - \alpha m
$$

where:

- $\beta$ is momentum coefficient  
- $\alpha$ is learning rate  

---

## 8. Soft Thresholding (L1 Regularization)

After gradient update, apply soft-thresholding:

$$
w_j =
\text{sign}(w_j) \cdot \max(|w_j| - \alpha \lambda_1, 0)
$$

This enforces sparsity by shrinking small weights to zero.

---

## 9. Initialization

Weights and momentum are initialized as:

$$
w = 0, \quad m = 0
$$

---

## 10. Iterative Optimization

For each epoch:

Shuffle dataset  

Generate mini-batches  

For each batch:

Compute predictions

$$
\hat{y}_{batch} = X_{batch} w
$$

Compute gradient

$$
\nabla_w = \frac{1}{B} X_{batch}^T (X_{batch} w - y_{batch}) + \lambda_2 w
$$

Update momentum

$$
m = \beta m + (1 - \beta)\nabla_w
$$

Update weights

$$
w = w - \alpha m
$$

Apply soft-thresholding

$$
w \leftarrow \text{SoftThreshold}(w)
$$

---

## 11. Full Loss Computation

After each epoch, compute:

$$
L =
\frac{1}{2N} \|Xw - y\|_2^2
+ \lambda_1 \|w\|_1
+ \frac{\lambda_2}{2} \|w\|_2^2
$$

---

## 12. Convergence Criterion

Training stops early if:

$$
|L_{\text{current}} - L_{\text{previous}}| < \text{tol}
$$

---

## 13. Effect of Regularization

- $\lambda_1 \uparrow$ → more sparsity (feature selection)  
- $\lambda_2 \uparrow$ → smoother, smaller weights  
- Combination balances **bias–variance tradeoff**

---

## 14. Prediction

After training, predictions are:

$$
\hat{y} = Xw
$$

---

## 15. Algorithm Summary

Initialize

$$
w = 0, \quad m = 0
$$

Repeat for each epoch:

- Shuffle data  
- Generate mini-batches  
- Compute predictions  
- Compute gradient  
- Update momentum  
- Update weights  
- Apply soft-thresholding  
- Compute loss  

Stop if convergence criterion is met.

---

## 16. Final Insight

Elastic Net combines the strengths of:

- **Lasso** → feature selection  
- **Ridge** → stability and smoothness  

$$
\text{Elastic Net} = \text{L1} + \text{L2}
$$

It is especially useful when:

- Features are correlated  
- Sparse but stable solutions are required  

In [1]:
class ElasticNetRegression:
    """
        Elastic Net Regression using Mini-Batch Gradient Descent with Momentum.
    
        This implementation combines:
        - L1 regularization (Lasso) for sparsity
        - L2 regularization (Ridge) for stability
        - Momentum for smoother and faster convergence
    
        Parameters
        ----------
        alpha : float, default=0.01
            Learning rate for gradient descent.
        beta : float, default=0.9
            Momentum coefficient (typically between 0 and 1).
        batch_size : int, default=10
            Size of mini-batches used during training.
        tol : float, default=1e-5
            Tolerance for early stopping based on loss improvement.
        add_bias : bool, default=True
            Whether to include a bias (intercept) term.
        epochs : int, default=10
            Number of passes over the dataset.
        lambda_l1 : float, default=1
            L1 regularization strength (controls sparsity).
        lambda_l2 : float, default=1
            L2 regularization strength (controls weight shrinkage).
        verbose : bool, default=False
            If True, prints loss at each epoch.
    
        Attributes
        ----------
        weights : ndarray of shape (n_features,)
            Learned model parameters.
        momentum : ndarray of shape (n_features,)
            Momentum vector used for updates.
                
    """


    def __init__(self,alpha=0.01, beta=0.9, batch_size = 10, tol =1e-5, add_bias =True, epochs=10, lambda_l1=1,lambda_l2=1,verbose=False ):
        
        self.alpha = alpha
        self.beta = beta
        self.batch_size = batch_size
        self.add_bias = add_bias
        self.epochs = epochs
        self.lambda_l1 = lambda_l1
        self.lambda_l2 = lambda_l2
        self.tol = tol
        self.verbose = verbose
        

        # Model parameters (initialized during training)
        self.weights = None
        self.momentum = None


    def _shuffle_indices(self,N):
        """
        Shuffle dataset indices.

        Parameters
        ----------
        N : int
            Number of samples.

        Returns
        -------
        ndarray
            Shuffled indices.
        """
        indices = np.arange(N)
        np.random.shuffle(indices)

        return indices

    def _batch_generator(self, shuffled_indices , batch_size):
        """
        Generate mini-batch indices.

        Parameters
        ----------
        shuffled_indices : ndarray
            Shuffled indices of the dataset.
        batch_size : int
            Size of each mini-batch.

        Yields
        ------
        ndarray
            Indices corresponding to a mini-batch.
        """

        for start in range(0,len(shuffled_indices),batch_size):
            batch_index = shuffled_indices[start:start+batch_size]

            yield batch_index


    def _soft_threshold(self,weight):
        """
        Apply soft-thresholding for L1 regularization.

        This operation shrinks weights towards zero and sets small values to zero.

        Parameters
        ----------
        weight : ndarray
            Weight vector.

        Returns
        -------
        ndarray
            Thresholded weight vector.
        """
        return np.sign(weight) * np.maximum( np.abs(weight) - self.alpha * self.lambda_l1 ,0)

        
    def fit(self,X,y):
        """
        Train the Elastic Net model using mini-batch gradient descent with momentum.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Training feature matrix.
        y : ndarray of shape (n_samples,)
            Target values.

        Notes
        -----
        - Uses L2 penalty in gradient computation.
        - Applies L1 penalty using soft-thresholding.
        - Stops early if loss improvement is below tolerance.
        """
        # Convert inputs to numpy arrays
        X = np.asarray(X)
        y = np.asarray(y)

        y = y.reshape(-1)  # Ensure y is 1D

        # Ensure X is 2D
        if X.ndim==1:
            X = X.reshape(-1,1)

        # Add bias column if required
        N , D = X.shape

        if self.add_bias :
            X = np.column_stack((np.ones(N),X))
            D +=1
            
        # Initialize weights and momentum

        self.weights =  np.zeros(D)
        self.momentum = np.zeros(D)

        loss_previous = np.inf

        # Training loop
        for i in range(self.epochs):
            # Shuffle data at each epoch
            shuffled_indices = self._shuffle_indices(N)

            all_indexes = self._batch_generator(shuffled_indices,self.batch_size)
            
            # Generate mini-batches
            while True :
                try :
                    indices = next(all_indexes)
                except StopIteration:
                    break
                    
                # Extract mini-batch
                X_batch , y_batch = X[indices] , y[indices]

                # Compute predictions
                y_hat = X_batch @ self.weights

                # Compute gradient (MSE + L2 regularization)
                gradient = (X_batch.T @ (y_hat-y_batch))/len(indices) 
                gradient[1:] += self.weights[1:] * self.lambda_l2

                # Update momentum
                self.momentum = self.beta * self.momentum + (1-self.beta) * gradient

                # Update weights
                self.weights -= self.alpha * self.momentum

                # Apply L1 regularization via soft-thresholding (excluding bias)
                self.weights[1:] = self._soft_threshold(self.weights[1:])


            # Compute full loss (MSE + L1 + L2)
            loss_current = (0.5 * np.mean((y- X @ self.weights)**2) + 
                            self.lambda_l1 * np.sum(np.abs(self.weights[1:])) + 
                            0.5 * self.lambda_l2 * np.linalg.norm(self.weights[1:], ord=2) **2)

            if self.verbose :
                print(f'The loss in epoch {i} is {loss_current}')
                
            
            # Early stopping condition
            if abs(loss_current - loss_previous) < self.tol:
                print(f"Stopping early at epoch {i}: improvement < {self.tol}")
                break
                

            loss_previous = loss_current

        print(f'The weights of the model: {self.weights}')


    def predict(self,X):
        """
        Predict target values using the trained model.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Input feature matrix.

        Returns
        -------
        ndarray of shape (n_samples,)
            Predicted values.
        """
 
        X = np.asarray(X)

        # Ensure X is 2D
        if X.ndim==1:
            X = X.reshape(-1,1)

        
        N = len(X)

        # Add bias if required
        if self.add_bias :
            X = np.column_stack((np.ones(N),X))

        # Compute predictions
        return X @ self.weights
               



## 1. Objective

The goal is to analyze how **Elastic Net regularization** affects model weights.

Elastic Net combines:

- **L1 regularization (Lasso)** → promotes sparsity  
- **L2 regularization (Ridge)** → stabilizes weights  

Study the effect of varying:

$$
\lambda_{L1}, \quad \lambda_{L2}
$$

on learned weights.

---

## 2. Dataset


- First two features are **relevant**
- Remaining features are **noise**

---






In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Samples and features
N = 100
D = 6

# Generate features
X = np.random.randn(N, D)

# True weights (only first 2 features are important)
true_weights = np.array([5, -3, 0, 0, 0, 0])

# Generate target with noise
y = X @ true_weights + np.random.randn(N) * 0.5

In [3]:
settings = [
    {"lambda_l1": 0, "lambda_l2": 0},       # No regularization (Linear Regression Least Squares)
    {"lambda_l1": 0.1, "lambda_l2": 0},     # Lasso
    {"lambda_l1": 0, "lambda_l2": 1},       # Ridge
    {"lambda_l1": 0.5, "lambda_l2": 0.5},   # Elastic Net
    {"lambda_l1": 1, "lambda_l2": 5},       # Strong regularization
]

results = []

for setting in settings:
    model = ElasticNetRegression(
        alpha=0.01,
        epochs=200,
        batch_size=20,
        add_bias=False,
        lambda_l1=setting["lambda_l1"],
        lambda_l2=setting["lambda_l2"],
        verbose=False
    )

    model.fit(X, y)
    print('\n')

    results.append({
        "lambda_l1": setting["lambda_l1"],
        "lambda_l2": setting["lambda_l2"],
        "weights": np.round(model.weights,2)
    })

df_results = pd.DataFrame(results)


Stopping early at epoch 159: improvement < 1e-05
The weights of the model: [ 4.92175033 -3.03045348 -0.01459872  0.0476342   0.07799491  0.03194051]


Stopping early at epoch 147: improvement < 1e-05
The weights of the model: [ 4.92089899 -2.91376733  0.          0.          0.          0.        ]


Stopping early at epoch 132: improvement < 1e-05
The weights of the model: [ 4.87086032 -1.38403123  0.14561788  0.00753743 -0.01551938  0.17033439]


Stopping early at epoch 123: improvement < 1e-05
The weights of the model: [ 4.88038136 -1.56908451  0.          0.         -0.          0.        ]


Stopping early at epoch 125: improvement < 1e-05
The weights of the model: [ 4.8674107  -0.28263335  0.         -0.         -0.          0.        ]




---

In [4]:
df_results

,lambda_l1,lambda_l2,weights
0,0.0,0.0,"[4.92, -3.03, -0.01, 0.05, 0.08, 0.03]"
1,0.1,0.0,"[4.92, -2.91, 0.0, 0.0, 0.0, 0.0]"
2,0.0,1.0,"[4.87, -1.38, 0.15, 0.01, -0.02, 0.17]"
3,0.5,0.5,"[4.88, -1.57, 0.0, 0.0, -0.0, 0.0]"
4,1.0,5.0,"[4.87, -0.28, 0.0, -0.0, -0.0, 0.0]"


## 3. Observations

### Case 1: No Regularization

- All weights are non-zero  
- Model fits noise  
- High variance  

---

### Case 2: L1 Regularization (Lasso-like)

- Many weights become exactly zero  

$$
w_j = 0
$$

- Performs **feature selection**  

---

### Case 3: L2 Regularization (Ridge-like)

- Most weights shrink smoothly  

$$
w_j \to 0 \quad \text{but not exactly zero}
$$

- Improves stability  

---

### Case 4: Elastic Net 

- Combines both effects:
  - Sparsity from L1  
  - Smooth shrinkage from L2  

---

### Case 5: Strong Regularization

- Most weights become very small  

$$
w \approx 0
$$

- Model underfits  

---

## 4. Geometric Interpretation

Elastic Net constraint:

$$
\lambda_{L1} \|w\|_1 + \lambda_{L2} \|w\|_2^2
$$

- L1 → diamond constraint → sparsity  
- L2 → circular constraint → smooth shrinkage  

Elastic Net → hybrid constraint region  

---

## 5. Conclusion

- **L1 (Lasso)** → sparse model, feature selection  
- **L2 (Ridge)** → stable model, no sparsity  
- **Elastic Net** → balance between sparsity and stability  

Elastic Net is particularly useful when:

- Features are **correlated**
- We want **automatic feature selection**
- We need a **stable model**

---
